In [3]:
import time
from time import sleep, monotonic
import datetime
import numpy as np
import matplotlib.pyplot as plt
import sys
import pyvisa
import qcodes as qc
from qcodes.dataset import Measurement
from qcodes.dataset import do0d
from qcodes.dataset.experiment_container import new_experiment, load_experiment_by_name
from qcodes.dataset.plotting import plot_by_id
from qcodes.dataset.data_set import load_by_id, load_by_counter
from qcodes import initialise_or_create_database_at, new_data_set, new_experiment
from qcodes.station import Station
initialise_or_create_database_at("./2026-03-10_SNSPD4.db")
from functions import osc_set_standard, osc_check_standard, capture_trace, capture_trace_simple
import snspd
params = snspd.snspd()
# Set up experiment
exp_name = 'SNSPD4_23_03_2026'
sample_name = '00'

try:
    exp = qc.load_experiment_by_name(exp_name, sample=sample_name)
    print('Experiment loaded. Last ID no:', exp.last_counter)
except ValueError:
    exp = new_experiment(exp_name, sample_name)
    print('Started new experiment')

Experiment loaded. Last ID no: 464


In [ ]:
verticalsnap = snap['instruments']['mso5']['submodules']['channels']['channels']['mso5_ch1']['parameters'].keys()

In [ ]:
horizontalsnap = snap['instruments']['mso5']['parameters'].keys()

In [2]:
station = Station(config_file="friesland.yaml")

In [3]:
yoko = station.load_instrument("yoko", revive_instance=True)

Connected to: YOKOGAWA GS210 (serial:91T928105, firmware:2.02) in 0.09s


In [5]:
laser = station.load_instrument("laser", revive_instance=True)

2026-04-10 15:29:39,144 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ D:\SNSPD\SNSPD2\PPCL550.py:4: QCoDeSDeprecationWarning: The `qcodes.utils.helpers` module is deprecated. Please consult the api documentation at https://microsoft.github.io/Qcodes/api/index.html for alternatives.
  from qcodes.utils.helpers import create_on_off_val_mapping

2026-04-10 15:29:39,290 ¦ qcodes.instrument.instrument_base.com.visa ¦ ERROR ¦ visa ¦ _connect_and_handle_error ¦ 222 ¦ [laser(PPCL550)] Could not connect at ASRL13
Traceback (most recent call last):
  File "C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\qcodes\instrument\visa.py", line 218, in _connect_and_handle_error
    visa_handle, visabackend, resource_manager = self._open_resource(
                                                 ~~~~~~~~~~~~~~~~~~~^
        address, visalib
        ^^^^^^^^^^^^^^^^
    )
    ^
  File "C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\qcodes\instrument\visa.py", line 246, in _open_resou

VisaIOError: VI_ERROR_RSRC_BUSY (-1073807246): The resource is valid, but VISA cannot currently access it.

In [4]:
yoko.current()

6e-06

In [18]:
params = snspd4.snspd4()

In [11]:
snap = station.snapshot(update=True)

In [12]:
horizontalsnap = snap['instruments']['mso5']['parameters'].keys()

KeyError: 'mso5'

In [13]:
load_by_id(params.system_dark_counts_id).snapshot['station']['instruments']['mso5']['parameters']['horizontal_samplerate']

{'__class__': 'qcodes.parameters.parameter.Parameter',
 'full_name': 'mso5_horizontal_samplerate',
 'value': 625000000.0,
 'raw_value': '625.0000E+6',
 'ts': '2026-04-10 18:58:58',
 'instrument': 'MSO5.MSO5',
 'instrument_name': 'mso5',
 'name': 'horizontal_samplerate',
 'label': 'horizontal_samplerate',
 'validators': [],
 'inter_delay': 0,
 'unit': 'Sa/s',
 'post_delay': 0}

In [14]:
load_by_id(params.att_screw_calibration_id).snapshot['station']['instruments']['mso5']['parameters']['horizontal_samplerate']

KeyError: 'mso5'

In [16]:
initial = 8e-6
currents = np.arange(6e-6, 12e-6, 0.5e-6)
start = currents[0]
if initial >= start: 
    ramp = np.arange(initial, start-0.1e-6, -0.1e-6) # ramp down 
elif initial < start: 
    ramp = np.arange(initial, start+0.1e-6, 0.1e-6) # ramp up

for r in ramp: 
    curr_set = round(r, 9)
    print(curr_set)
    time.sleep(0.5)


8e-06
7.9e-06
7.8e-06
7.7e-06
7.6e-06
7.5e-06
7.4e-06
7.3e-06
7.2e-06
7.1e-06
7e-06
6.9e-06
6.8e-06
6.7e-06
6.6e-06
6.5e-06
6.4e-06
6.3e-06
6.2e-06
6.1e-06
6e-06


In [9]:
r = 5.69
round(r, 2)

5.69

In [ ]:
def current_sweep(yoko, dmm, currents, station=None):
    
    # # Update experiment snapshot 
    # update_station(station)
    
    # Ramping to start current 
    start = currents[0]
    if yoko.current() >= start: 
        ramp = np.arange(initial, start-0.1e-6, -0.1e-6) # ramp down 
    elif yoko.current() < start: 
        ramp = np.arange(initial, start+0.1e-6, 0.1e-6) # ramp up

    print(f'Ramping current from {yoko.current()}A to {start}A')

    for r in ramp: 
        yoko.current(round(r, 9)) # Set the current 
        time.sleep(0.5)
    
    # Initialize measurement 
    meas = Measurement()
    meas.register_custom_parameter("ratio")
    meas.register_parameter(yoko.current)
    meas.register_parameter(dmm.volt)
    # meas.register_custom_parameter("T_MXC", label="mK")


    with meas.run() as datasaver:
        print(datasaver.run_id)
        for i in currents:
            yoko.current(i)
            time.sleep(2)
            if i == 0: 
                ratio = 0
            else: 
                ratio = dmm.volt()/yoko.current()
            
            # if tc.channels['MXC-flange'].measure()['temperature'][-1] == None:
            #     temp = -1
            # else: 
            #     temp = tc.channels['MXC-flange'].measure()['temperature'][-1]*1e3
            
            datasaver.add_result(('ratio',ratio),
                                (dmm.volt, dmm.volt()),
                                (yoko.current, yoko.current()))
                                # ("T_MXC", temp))
            time.sleep(0.1)

    # Ramping down
    print('Ramping down')
    for i in currents[::-1]: 
        yoko.current(i)
        time.sleep(0.1)

    print(f'Current is {yoko.current()}')
    print('Finished!')
